# DiffusionNet Ortodontik Landmark - Colab GPU

Bu notebook 23 landmarkli ortodontik 3B yuz datasetini DiffusionNet ile GPU uzerinde egitir ve Average Localization Error (ALE) raporlar.

Runtime menusu: `Runtime > Change runtime type > GPU` secili olmali.

In [ ]:
import torch, os, platform
print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 1. Drive bagla ve kodu hazirla

Dataset GitHub'a eklenmedigi icin Google Drive'dan okunur. Kod repodan klonlanir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf comparative-study
!git clone https://github.com/eckdev/comparative-study.git
%cd /content/comparative-study/diffusion_net_orthodontic_comparison
!python -m pip install -q -r requirements.txt

## 2. Veri yollarini ayarla

`DATA_ROOT` mevcut `data/dataset` klasorunu gostermeli. Transform klasoru yoksa `USE_TRANSFORMS = False` yap.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/orthodontic/data/dataset')
SPLITS_JSON = Path('/content/comparative-study/shared_splits/orthodontic_180_60_60_seed42.json')
TRANSFORM_DIR = Path('/content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801')
RUN_ROOT = Path('/content/drive/MyDrive/orthodontic/diffusion_runs')
USE_TRANSFORMS = TRANSFORM_DIR.exists()

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT exists:', DATA_ROOT.exists(), DATA_ROOT)
print('SPLITS_JSON exists:', SPLITS_JSON.exists(), SPLITS_JSON)
print('TRANSFORM_DIR exists:', TRANSFORM_DIR.exists(), TRANSFORM_DIR)
print('RUN_ROOT:', RUN_ROOT)

## 3. Dataset kontrolu

In [ ]:
import glob
ply_count = len(glob.glob(str(DATA_ROOT / 'Class*' / '*' / '*.ply')))
txt_count = len(glob.glob(str(DATA_ROOT / 'Class*' / 'Class*-Landmark' / '*' / '*.txt')))
print('PLY:', ply_count)
print('Landmark TXT:', txt_count)
assert ply_count > 0 and txt_count > 0, 'Dataset yolu yanlis gorunuyor.'

## 4. Pratik GPU kosusu

Bu preset T4/L4 gibi Colab GPU'larda paper metodolojisine sadik ama daha uygulanabilir bir ayardir. Yerel CPU kosusundaki `2048` noktadan daha yuksek `4096` nokta kullanir.

In [ ]:
import subprocess, shlex

run_dir = RUN_ROOT / 'diffusionnet_maskseg_xyz_p4096_k64_w128_b6_e120'
cmd = [
    'python', 'run_orthodontic_diffusion.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(run_dir),
    '--surface-points', '4096',
    '--k-eig', '64',
    '--epochs', '120',
    '--patience', '25',
    '--width', '128',
    '--blocks', '6',
    '--mlp-hidden-dims', '256',
    '--loss-mode', 'mask_bce',
    '--mask-radius', '3.5',
    '--input-features', 'xyz',
    '--postprocess', 'softmax',
    '--lr', '0.001',
    '--device', 'auto',
]
if USE_TRANSFORMS:
    cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

## 5. Paper'a daha yakin full-mesh kosu

Bu hucreyi A100/High-RAM gibi daha guclu GPU varsa calistir. T4'te VRAM veya sure limiti nedeniyle zorlanabilir.

In [ ]:
import subprocess, shlex

paper_run_dir = RUN_ROOT / 'diffusionnet_paper_like_fullmesh_xyz_mask_r3p5_w384_b12_e200'
paper_cmd = [
    'python', 'run_orthodontic_diffusion.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(paper_run_dir),
    '--use-mesh-vertices',
    '--k-eig', '128',
    '--epochs', '200',
    '--patience', '35',
    '--width', '384',
    '--blocks', '12',
    '--mlp-hidden-dims', '768',
    '--loss-mode', 'mask_bce',
    '--mask-radius', '3.5',
    '--input-features', 'xyz',
    '--postprocess', 'softmax',
    '--lr', '0.001',
    '--device', 'auto',
]
if USE_TRANSFORMS:
    paper_cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print(' '.join(shlex.quote(x) for x in paper_cmd))
# Uzun kosu icin asagidaki satirin yorumunu kaldir:
# subprocess.run(paper_cmd, check=True)

## 6. Sonuclari oku

In [ ]:
import json
metrics_path = run_dir / 'metrics.json'
metrics = json.loads(metrics_path.read_text())
summary = metrics['diffusionnet_heatmap']
print('Run:', run_dir)
print('ALE:', round(summary['ale'], 4))
print('Median:', round(summary['median'], 4))
print('Std:', round(summary['std'], 4))
print('Max:', round(summary['max'], 4))

## 7. Alternatif post-processing degerlendirmesi

Modeli yeniden egitmeden farkli aktivasyon-koordiant donusumlerini deneyebilirsin.

In [ ]:
eval_dir = RUN_ROOT / 'diffusionnet_maskseg_xyz_p4096_eval_topk10'
eval_cmd = [
    'python', 'run_orthodontic_diffusion.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(eval_dir),
    '--surface-points', '4096',
    '--k-eig', '64',
    '--width', '128',
    '--blocks', '6',
    '--mlp-hidden-dims', '256',
    '--loss-mode', 'mask_bce',
    '--mask-radius', '3.5',
    '--input-features', 'xyz',
    '--postprocess', 'topk_softmax',
    '--refine-topk', '10',
    '--refine-temperature', '1.0',
    '--evaluate-only',
    '--model-path', str(run_dir / 'best_model.pth'),
    '--device', 'auto',
]
if USE_TRANSFORMS:
    eval_cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print(' '.join(shlex.quote(x) for x in eval_cmd))
subprocess.run(eval_cmd, check=True)